![CCAI 9012 course banner](../../../docs/figs/brand_identity/banner.png)

# Literature review: evidence-based comparison across multiple papers

Suppose you need to compare methods and findings across a small reading list. Rather than copy four abstracts into notes, ask one structured question of each local PDF, then reopen candidate pages before using the comparison.

## What you will learn

- Locate text in a PDF and retain its page number.
- Extract the same four fields from each paper with one DeepSeek call per PDF.
- Read a comparison table while tracking missing or uncertain evidence.
- Verify model summaries against the source pages.

![Research papers and a comparison question lead to page-linked evidence notes and source verification.](../../../docs/figs/literature_evidence_matrix.svg)


## Step 1 — Read local PDFs

The default run processes at most four repository PDFs. Each PDF sends one bounded excerpt collection to DeepSeek. A page number below is a **candidate lookup location**, not a verified citation.


In [3]:
from IPython.display import display
import pandas as pd
from ccai9012 import llm_utils
from ccai9012.paths import STARTER_KITS_DIR, ensure_output_dir

EXAMPLE_ROOT = STARTER_KITS_DIR / "2_llm_structure_output" / "lit_review"
OUTPUT_DIR = ensure_output_dir(EXAMPLE_ROOT)
pdf_paths = sorted((EXAMPLE_ROOT / "data").glob("*.pdf"))[:4]
assert len(pdf_paths) == 4, "Expected four local research PDFs"
for path in pdf_paths:
    pages = llm_utils.extract_pdf_pages(path)
    print(f"{path.name}: {len(pages)} PDF pages")


Ignoring wrong pointing object 624 0 (offset 0)
Ignoring wrong pointing object 625 0 (offset 0)
Ignoring wrong pointing object 628 0 (offset 0)
Ignoring wrong pointing object 629 0 (offset 0)
Ignoring wrong pointing object 663 0 (offset 0)
Ignoring wrong pointing object 664 0 (offset 0)


1-s2.0-S1353829223001867-main.pdf: 11 PDF pages
DeepSOCIAL.pdf: 29 PDF pages
Monitoring COVID-19 social distancing.pdf: 10 PDF pages
Small public space vitality analysis and evaluation.pdf: 14 PDF pages


## Step 2 — Extract comparable fields

DeepSeek receives candidate excerpts with page numbers and returns four fields. `N/A` means the supplied context was insufficient. Keep the candidate passage beside each model answer so a reader can challenge it.


In [4]:
EXPECTED_FIELDS = ["Problem", "Research Gap", "Methodology", "Key Results"]
llm = llm_utils.initialize_llm(model="deepseek-chat", temperature=0)
rows = []
for pdf_path in pdf_paths:
    rows.extend(llm_utils.compare_pdf_with_llm(pdf_path, llm, EXPECTED_FIELDS))
evidence_matrix = pd.DataFrame(rows)
evidence_matrix.to_csv(OUTPUT_DIR / "literature_evidence_matrix.csv", index=False)
display(evidence_matrix.head(8))
print(f"requests={len(pdf_paths)}; evidence_rows={len(evidence_matrix)}")


Ignoring wrong pointing object 624 0 (offset 0)
Ignoring wrong pointing object 625 0 (offset 0)
Ignoring wrong pointing object 628 0 (offset 0)
Ignoring wrong pointing object 629 0 (offset 0)
Ignoring wrong pointing object 663 0 (offset 0)
Ignoring wrong pointing object 664 0 (offset 0)


,paper,extracted_field,value,retrieved_passage,source_location,uncertainty
0,1-s2.0-S1353829223001867-main.pdf,Problem,Not enough studies have examined how specific ...,Health & Place 85 (2024) 103149 Available onli...,PDF page 1,Candidate source page; verify the claim in the...
1,1-s2.0-S1353829223001867-main.pdf,Research Gap,"To our knowledge, no existing studies have use...",Health and Place 85 (2024) 103149 9 2019). Our...,PDF page 9,Candidate source page; verify the claim in the...
2,1-s2.0-S1353829223001867-main.pdf,Methodology,This study uses deep learning-based methods to...,Health and Place 85 (2024) 103149 2 In compari...,PDF page 2,Candidate source page; verify the claim in the...
3,1-s2.0-S1353829223001867-main.pdf,Key Results,The results show that interaction with semi-fi...,Health and Place 85 (2024) 103149 7 observed t...,PDF page 7,Candidate source page; verify the claim in the...
4,DeepSOCIAL.pdf,Problem,The COVID-19 pandemic has created challenges f...,"Appl. Sci. 2020, 10, 7514 3 of 29 essential ne...",PDF page 3,Candidate source page; verify the claim in the...
5,DeepSOCIAL.pdf,Research Gap,Existing research on people detection using AI...,"Appl. Sci. 2020, 10, 7514 4 of 29 More details...",PDF page 4,Candidate source page; verify the claim in the...
6,DeepSOCIAL.pdf,Methodology,The paper proposes a methodology including mod...,"Appl. Sci. 2020, 10, 7514 14 of 29 The camera ...",PDF page 14,Candidate source page; verify the claim in the...
7,DeepSOCIAL.pdf,Key Results,The DeepSOCIAL model variants achieved high pr...,"Appl. Sci. 2020, 10, 7514 4 of 29 More details...",PDF page 4,Candidate source page; verify the claim in the...


requests=4; evidence_rows=16


### Pause and check one claim

Choose a non-`N/A` row, open its PDF at `source_location`, and compare the claim with the whole paragraph. The retrieval score only selects a candidate page; it does not validate the model answer.


In [5]:
comparison = evidence_matrix.pivot(index="paper", columns="extracted_field", values="value").reset_index()
comparison = comparison.reindex(columns=["paper"] + EXPECTED_FIELDS)
comparison.to_csv(OUTPUT_DIR / "literature_comparison.csv", index=False)
display(comparison)


extracted_field,paper,Problem,Research Gap,Methodology,Key Results
0,1-s2.0-S1353829223001867-main.pdf,Not enough studies have examined how specific ...,"To our knowledge, no existing studies have use...",This study uses deep learning-based methods to...,The results show that interaction with semi-fi...
1,DeepSOCIAL.pdf,The COVID-19 pandemic has created challenges f...,Existing research on people detection using AI...,The paper proposes a methodology including mod...,The DeepSOCIAL model variants achieved high pr...
2,Monitoring COVID-19 social distancing.pdf,Human detection using visual surveillance syst...,Manual methods for identifying unusual activit...,The proposed framework utilizes the YOLO v3 ob...,YOLO v3 with Deepsort tracking scheme displaye...
3,Small public space vitality analysis and evalu...,It is not comprehensive to quantify pedestrian...,Some existing methods mainly focused on quanti...,The study collects a video dataset of small pu...,The guidance of the expert assessment method e...


### How to read the comparison

Scan one column across papers, then return to the matching rows in `evidence_matrix` for pages and passages. Similar wording may hide different definitions or populations. A blank field calls for reading more context, not guessing.


In [6]:
# The saved CSVs are optional review artifacts; rerunning this notebook calls DeepSeek again.
print(OUTPUT_DIR / "literature_evidence_matrix.csv")


/Users/kanxuanhe/Document/250421_AI_course/ccai9012/starter_kits/2_llm_structure_output/lit_review/output/literature_evidence_matrix.csv


## Reflect

Which two papers can fairly be compared on methodology, and which need a closer reading? Record corrected claims and exact source pages before citing any paper.
